# Extract




In [66]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, Date, String, Float

url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
df = pd.read_csv(url)
df

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429430,ZWE,Africa,Zimbabwe,2024-07-31,266386.0,0.0,0.0,5740.0,0.0,0.0,...,30.7,36.79,1.7,61.49,0.57,16320539,NaN,NaN,NaN,NaN
429431,ZWE,Africa,Zimbabwe,2024-08-01,266386.0,0.0,0.0,5740.0,0.0,0.0,...,30.7,36.79,1.7,61.49,0.57,16320539,NaN,NaN,NaN,NaN
429432,ZWE,Africa,Zimbabwe,2024-08-02,266386.0,0.0,0.0,5740.0,0.0,0.0,...,30.7,36.79,1.7,61.49,0.57,16320539,NaN,NaN,NaN,NaN
429433,ZWE,Africa,Zimbabwe,2024-08-03,266386.0,0.0,0.0,5740.0,0.0,0.0,...,30.7,36.79,1.7,61.49,0.57,16320539,NaN,NaN,NaN,NaN


# Transform

In [63]:
# Checking columns to determine which are the most important
columns = df.columns.tolist()
for column in columns:
  print(f"{column},")

iso_code,
continent,
location,
date,
total_cases,
new_cases,
new_cases_smoothed,
total_deaths,
new_deaths,
new_deaths_smoothed,
total_cases_per_million,
new_cases_per_million,
new_cases_smoothed_per_million,
total_deaths_per_million,
new_deaths_per_million,
new_deaths_smoothed_per_million,
reproduction_rate,
icu_patients,
icu_patients_per_million,
hosp_patients,
hosp_patients_per_million,
weekly_icu_admissions,
weekly_icu_admissions_per_million,
weekly_hosp_admissions,
weekly_hosp_admissions_per_million,
total_tests,
new_tests,
total_tests_per_thousand,
new_tests_per_thousand,
new_tests_smoothed,
new_tests_smoothed_per_thousand,
positive_rate,
tests_per_case,
tests_units,
total_vaccinations,
people_vaccinated,
people_fully_vaccinated,
total_boosters,
new_vaccinations,
new_vaccinations_smoothed,
total_vaccinations_per_hundred,
people_vaccinated_per_hundred,
people_fully_vaccinated_per_hundred,
total_boosters_per_hundred,
new_vaccinations_smoothed_per_million,
new_people_vaccinated_smoot

In [67]:
# Simplifying dataset by filtering columns
columns_to_keep = [
    "iso_code",
    "date",
    "new_cases",
    "new_deaths",
    "new_tests",
    "new_vaccinations",
    "stringency_index",
]

df = df[columns_to_keep]

# Filtering to get only registers from Brazil
df = df[df['iso_code'] == "BRA"]

# Removing iso_code column cause all of its records are the same
df = df.drop('iso_code', axis=1)

# Convert the 'date' column to datetime.date objects,
# so it will be properly interpreted by Load tools
df['date'] = pd.to_datetime(df['date']).dt.date

df.head()

,date,new_cases,new_deaths,new_tests,new_vaccinations,stringency_index
50234,2020-01-05,0.0,0.0,NaN,NaN,0.0
50235,2020-01-06,0.0,0.0,NaN,NaN,0.0
50236,2020-01-07,0.0,0.0,NaN,NaN,0.0
50237,2020-01-08,0.0,0.0,NaN,NaN,0.0
50238,2020-01-09,0.0,0.0,NaN,NaN,0.0


# Load

In [68]:
conn = sqlite3.connect('./brazil_covid.db')

engine = create_engine('sqlite:///brazil_covid.db')

db_schema = {
    'date': Date,
    'new_cases': Float,
    'new_deaths': Float,
    'new_tests': Float,
    'new_vaccinations': Float,
    'stringency_index': Float,
}

df.to_sql(
    name='brazil_covid_data',
    con=engine,
    if_exists='replace',
    index=False,
    dtype=db_schema
)

conn.commit()
conn.close()

print('Data has been successfully loaded into SQL database!')

Data has been successfully loaded into SQL database!
